# 27baa all-cost own-account error attribution

One frozen read-only account decomposition. No new policy.

In [1]:
import hashlib,json,os
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.analysis.etf_transfer_attribution import holdings_attribution
root=Path.cwd();card=root/'research/raw_27baa_all_cost_window_attribution_20261112';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();source=root/plan['source_card']
assert sha(source/'summary.json')==plan['source_summary_sha256']
assert sha(source/'verification_manifest.json')==plan['source_manifest_sha256']
qlib.init(provider_uri=str(root/'data/qlib/frozen_policy_raw_cashbook_20261015'),region='us',kernels=1)
objects,receipt=load_verified_recorder_objects(source,expected_recorder_id=plan['source_recorder_id'],artifacts=plan['source_artifacts'],expected_manifest_sha256=plan['source_manifest_sha256'])
frame=objects['inputs/frame'];prices=objects['inputs/raw_points'];schedule=objects['inputs/schedule']
assert len(frame)==944 and len(prices)==3778
raw_targets=frame.set_index('signal_date')[['soxl_weight','soxx_weight','cash_weight']]
all_daily={};all_windows={};annual={};summary_by_cost={}
for cost in (0,2,5):
    book=objects[f'native/raw_candidate27baa_{cost}'];bench=objects[f'native/raw_SOXX_{cost}']
    daily,segments=holdings_attribution(book,prices,schedule,raw_targets,initial_cash=100000)
    assert len(daily)==944 and len(segments)==944*4*2
    pd.testing.assert_series_equal(pd.to_datetime(daily.trade_date).reset_index(drop=True),pd.to_datetime(bench['authority'].trade_date).reset_index(drop=True),check_names=False)
    br=bench['authority'].strategy_return.to_numpy(float);sr=daily['return'].to_numpy(float)
    assert np.isfinite(np.r_[sr,br]).all() and (sr>-1).all() and (br>-1).all()
    seg=segments.pivot(index=['trade_date','slot'],columns='asset',values=['weight','asset_return'])
    assert len(seg)==944*4
    w1=seg[('weight','SOXL')].to_numpy(float);w2=seg[('weight','SOXX')].to_numpy(float)
    r1=seg[('asset_return','SOXL')].to_numpy(float);r2=seg[('asset_return','SOXX')].to_numpy(float)
    seg_cash=-(1-w1-w2)*r2;seg_selection=w1*(r1-r2)
    parts=pd.DataFrame({'trade_date':seg.index.get_level_values('trade_date'),'cash_arith':seg_cash,'selection_arith':seg_selection}).groupby('trade_date').sum()
    daily=daily.set_index('trade_date',drop=False).join(parts)
    daily['benchmark_return']=br
    daily['fee_arith']=daily.fee_contribution
    delta=sr-br;logex=np.log1p(sr)-np.log1p(br)
    scalar=np.divide(logex,delta,out=1/(1+br),where=np.abs(delta)>1e-12)
    daily['residual_arith']=delta-daily.cash_arith.to_numpy()-daily.selection_arith.to_numpy()-daily.fee_arith.to_numpy()
    for k in ['cash','selection','fee','residual']:
        daily[k+'_log']=daily[k+'_arith'].to_numpy(float)*scalar
    daily['relative_log']=logex
    maxerr=float(np.max(np.abs(daily[['cash_log','selection_log','fee_log','residual_log']].sum(axis=1).to_numpy()-logex)))
    assert maxerr<1e-9
    assert abs(float(np.exp(logex.sum()))-float((1+sr).prod()/(1+br).prod()))<1e-8
    windows=[]
    for i in range(819):
        g=daily.iloc[i:i+126]
        row={'start':str(g.trade_date.iloc[0].date()),'end':str(g.trade_date.iloc[-1].date()),'cost_bps':cost}
        for k in ['cash','selection','fee','residual','relative']:
            row[k+'_log']=float(g[k+'_log'].sum())
        row['winner']=row['relative_log']>0
        windows.append(row)
    windows=pd.DataFrame(windows)
    assert len(windows)==819
    for group in (True,False):
        g=windows.loc[windows.winner.eq(group)]
        summary_by_cost[f'{cost}_{"winner" if group else "loser"}']={'windows':len(g),'cash_negative':int(g.cash_log.lt(0).sum()),'selection_negative':int(g.selection_log.lt(0).sum()),'both_negative':int((g.cash_log.lt(0)&g.selection_log.lt(0)).sum()),'mean_cash_log':float(g.cash_log.mean()),'mean_selection_log':float(g.selection_log.mean()),'mean_fee_log':float(g.fee_log.mean()),'mean_residual_log':float(g.residual_log.mean())}
    annual[str(cost)]={str(y):{k:float(v) for k,v in daily.loc[daily.trade_date.dt.year.eq(y),['relative_log','cash_log','selection_log','fee_log','residual_log']].sum().items()} for y in (2023,2024,2025,2026)}
    all_daily[str(cost)]=daily;all_windows[str(cost)]=windows
    print('cost',cost,'windows',int(windows.winner.sum()),'closure',maxerr,flush=True)
assert [int(all_windows[str(c)].winner.sum()) for c in (0,2,5)]==[553,549,539]
losssets={c:set(all_windows[str(c)].loc[~all_windows[str(c)].winner,'start']) for c in (0,2,5)}
overlap={'all_three_losing':len(set.intersection(*losssets.values())),'0_and_5_losing':len(losssets[0]&losssets[5]),'5_only_vs_0':len(losssets[5]-losssets[0])}
report={'status':'read_only_27baa_all_cost_components_closed_no_policy','historical_research_passed':False,'new_fits':0,'new_accounts':0,'source_recorder_id':plan['source_recorder_id'],'daily_sessions_per_cost':944,'windows_per_cost':819,'group_summary':summary_by_cost,'annual_components':annual,'cross_cost_window_overlap':overlap,'limitations':['overlapping hindsight windows are descriptions, not independent market evidence or a trading switch','cash and selection terms are accounting components, not executable marginal actions','historical first receipt, full-minute risk and qualified forward remain unknown']}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 R.save_objects(frozen_plan=plan,source_readback=receipt,report=report,daily_components=all_daily,rolling_components=all_windows)
 rec=R.get_recorder();rid,eid=rec.id,str(rec.experiment_id)
local=card/'qlib_recorder'/eid/rid;names=['frozen_plan','source_readback','report','daily_components','rolling_components']
manifest={'recorder_id':rid,'entries':[{'path':(local/'artifacts'/k).relative_to(card).as_posix(),'sha256':sha(local/'artifacts'/k),'bytes':(local/'artifacts'/k).stat().st_size} for k in names]}
(card/'verification_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
summary={'experiment':plan['experiment'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str((card/'qlib_recorder').resolve()),'recorder_local_dir':str(local.resolve()),'plan_sha256':sha(card/'plan.json'),**report,'artifact_policy':{'policy':'slim_card_recorder_first','recorder_only_artifacts':names}}
(card/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False)+'\n')
check,rr=load_verified_recorder_objects(card,expected_recorder_id=rid,artifacts=['report','rolling_components'],expected_manifest_sha256=sha(card/'verification_manifest.json'))
assert check['report']==report
for cost in (0,2,5):pd.testing.assert_frame_equal(check['rolling_components'][str(cost)],all_windows[str(cost)])
(card/'readback_receipt.json').write_text(json.dumps({'recorder_id':rid,'summary_sha256':sha(card/'summary.json'),'manifest_sha256':sha(card/'verification_manifest.json'),'readback':rr},indent=2)+'\n')
print(json.dumps({'status':report['status'],'group_summary':summary_by_cost,'overlap':overlap,'recorder_id':rid},indent=2))

{
  "status": "read_only_27baa_all_cost_components_closed_no_policy",
  "group_summary": {
    "0_winner": {
      "windows": 553,
      "cash_negative": 217,
      "selection_negative": 106,
      "both_negative": 0,
      "mean_cash_log": 0.024200263618446516,
      "mean_selection_log": 0.04984259220885556,
      "mean_fee_log": -0.0009036386779337691,
      "mean_residual_log": 0.0038971051949288262
    },
    "0_loser": {
      "windows": 266,
      "cash_negative": 262,
      "selection_negative": 142,
      "both_negative": 138,
      "mean_cash_log": -0.046304884267577076,
      "mean_selection_log": 0.0005386551063207536,
      "mean_fee_log": -0.001179340875756979,
      "mean_residual_log": 0.0022382070200345173
    },
    "2_winner": {
      "windows": 549,
      "cash_negative": 213,
      "selection_negative": 105,
      "both_negative": 0,
      "mean_cash_log": 0.02473870169610389,
      "mean_selection_log": 0.05009416240277116,
      "mean_fee_log": -0.002808371513357

## Conclusion

**No stable SOXX excess.** Saved 27baa all-cost accounts close exactly to the native daily books. Among 0/2/5bp losing 126-session windows, cash opportunity is negative in 262/266, 265/270, and 272/280 respectively; average SOXL selection in each losing group is near zero or positive. The 266 losing 0bp windows remain losing at 2/5bp. These are overlapping hindsight accounting groups, not a trading switch. See `conclusion.md`, `summary.json`, and local/NAS readback receipts.